# FASTQ Format Validation
This report reflects the FASTQ files Harpy processed to identify obvious
[formatting issues](#fastq-details) that may require your attention.

In [ ]:
cat(format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'))

In [ ]:
library(dplyr)
library(DT)
library(scales)
library(tidyr)

In [ ]:
infile <- "test.tsv"
platform <- "haplotagging"

In [ ]:
data <- read.table(infile, header = T)
attention_df <- data %>% 
  mutate(allmissing = (reads == noBX)) %>% 
  select(4,5,7) %>% 
  rowwise() %>% 
  summarise(count = sum(c(badBX,badSamSpec, allmissing)))
attention <- sum(attention_df$count > 0)
bxnotlast <- sum(data$bxNotLast > 0)

In [ ]:
create_colored_box <- function(value, label, color = "steelblue", width = "200px", height = "90px") {
  .val <- formatC(value, format = "f", big.mark = ",", digits = 2, drop0trailing = T)
  sprintf(
    '<div style="background-color: %s; width: %s; height: %s; display: flex; flex-direction: column; align-items: center; justify-content: center; color: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); padding: 20px; box-sizing: border-box;"><div style="font-size: 14px; font-weight: normal; margin-bottom: 2px; margin-top: 13px; opacity: 0.9; text-transform: uppercase; letter-spacing: 1px;">%s</div><div style="font-size: 48px; font-weight: bold;">%s</div></div>',
    color, width, height, label, .val
    )
}

# Function to display boxes that wrap dynamically
show_boxes <- function(boxes, gap = "15px") {
  box_html <- paste(boxes, collapse = "\n")
  html_content <- sprintf('<div style="display: flex; flex-wrap: wrap; gap: %s; width: 100%%;">%s</div>', gap, box_html)
  IRdisplay::display_html(html_content)
}

In [ ]:
show_boxes(c(
  create_colored_box(nrow(data), "Files", "#aeaeaeff"),
  create_colored_box(attention, "Needs Review", ifelse(attention > 0, "#f6ab3c", "#68ae6bff")),
  create_colored_box(bxnotlast, "BX Not Last", ifelse(bxnotlast > 0, "#f6ab3c", "#68ae6bff"))
))

## Metrics
[Hover me](#fastq-metrics-def) to see the metric descriptions, or view them at the bottom of this report.

In [ ]:
datatable(
  data,
  escape = F,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  colnames = c("File", "Reads", "AxxCxxBxxDxx", "SAM Spec", "BX:Z last", "BX:Z tag"),
  options = list(
    paging = F,
    scrollX = F,
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "fastq_checks"))
  )
) %>%
formatStyle(
  columns = 3:ncol(data),
  # green for values <= 0, yellow for values > 0
  backgroundColor = styleInterval(0, c('#68ae6bff', '#f6ab3c')) 
) %>%
formatStyle(
  columns = 2,
  # yellow for values == 0, green for values > 0
  backgroundColor = styleInterval(0, c('#f6ab3c', '#68ae6bff')) 
)

:::{note} Metric Definitions
:label: fastq-metrics-def
:class: dropdown
:icon: false

| column        | 🟩 pass condition 🟩                                                        | 🟨 fail condition 🟨                                       |
|:--------------|:----------------------------------------------------------------------------|:-----------------------------------------------------------|
| **Format**    | **all** reads with `BX:Z` tag have properly formatted barcodes for their chemistry | **any** BX:Z barcodes have incorrectly formatted barcodes for their chemistry   |
| **SAM Spec**  | **all** reads have proper `TAG:TYPE:VALUE` comments                         | **any** reads have incorrectly formatted comments          |
| **BX:Z last** | **all** reads have `BX:Z` as final comment                                  | **at least 1 read** doesn’t have BX:Z tag as final comment |
| **BX:Z tag**  | **any** `BX:Z` tags present                                                 | **all** reads lack BX:Z tag                                |

:::

:::{note} Interpreting the data
:label: fastq-details
:class: dropdown
:icon: false

The `harpy validate fastq ...` command created a `validate.fastq.tsv` file in the specified
output directory that summarizes the results that are included in this report.
This file contains a tab-delimited table with the columns: `file`, `reads`, `noBX`, `badBX`, and `badSamSpec`.
These columns are defined below and the severities of their issues are given as colored emoji:

- ⬜ is minor
- 🔶 is moderate
- 🛑 is critical

file
: The name of the FASTQ file.

reads
: The total number of reads in the file.

noBX ⬜
: The number of reads that do not have a `BX:Z` tag in the read header.
If you expect all or some of your reads should have `BX:Z` tags, then further investigation is necessary.

badBX 🛑
: The barcode does is not in the proper `r platform` format.

badSamSpec 🛑
: The comments in the read header after the read ID do not conform to the `TAG:TYPE:VALUE` [SAM specification](https://samtools.github.io/hts-specs/SAMv1.pdf).

bxNotLast 🔶
: The `BX:Z:` tag in the FASTQ header is not the last comment. 
Only relevant for LEVIATHAN variant calling and can be ignored if not intending to call
structural variants with LEVIATHAN,otherwise the `BX:Z` tag must be the last comment in a read
header. `harpy align` will automatically move the `BX:Z` tag to the end of the alignment record.
:::

:::{note} Proper Read Headers
:class: dropdown
:icon: false

An example of a proper FASTQ read header is like the one below, where all comments
following the initial read ID (`@A00814...`) have the proper [SAM spec](https://samtools.github.io/hts-specs/SAMv1.pdf)
`TAG:TYPE:VALUE` format and among them is a `BX:Z` tag followed by a `AxxCxxBxxDxx`
formatted haplotagging barcode. 

The comments must be **TAB separated** because the `:Z` tag type allows a 
whitespace character for its value. 

**Example Header**

The read header below contains the read ID `@A00814...`,
the `BX:Z` haplotagging barcode tag, and two more comments `RX:Z` and `QX:Z` that both
adhere to the SAM specification. If using LEVIATHAN to call structural variants, the `BX:Z` tag
must be the last comment in the read header.
```
@A00814:267:HTMH3DRXX:2:1101:4580:1000	BX:Z:A02C01B11D46	RX:Z:GAAACGACCAACA+CGAACACGTTAGC	QX:Z:F,FFFFFFFFFFF+FF,FFF:FFFFFF
```

:::